# 02_raw_to_bronze

Convert Raw FHIR Bundles into structured Bronze Delta tables.

### Bronze responsibilities
- Read original JSON Bundle responses from Raw.
- Extract individual FHIR resources from `entry[].resource`.
- Preserve ingestion and API metadata.
- Write one Delta table per FHIR resource.

**Raw → Bronze only. Cleaning, deduplication, and SCD Type 2 are handled next.**


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 21, Finished, Available, Finished, False)

In [7]:
# Raw is the default lakehouse; Bronze is attached.
RAW_BASE_PATH = "Files/raw"
RAW_METADATA_PATH = "Files/raw_metadata"

BRONZE_BASE_PATH = "abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables"

RESOURCE_TYPES = [
    "patient",
    "encounter",
    "observation",
    "condition"
]

print(f"Raw path: {RAW_BASE_PATH}")
print(f"Bronze path: {BRONZE_BASE_PATH}")


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 27, Finished, Available, Finished, False)

Raw path: Files/raw
Bronze path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables


## 1. Read Raw FHIR Bundles

Raw contains the original paginated FHIR Bundle responses under each resource and ingestion date.


In [8]:
def read_raw_resource(resource_type: str):
    raw_path = f"{RAW_BASE_PATH}/{resource_type}"

    return (
        spark.read
        .option("multiLine", "true")
        .json(raw_path)
        .withColumn("source_file", F.input_file_name())
    )


patient_raw_df = read_raw_resource("patient")
patient_raw_df.printSchema()


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 28, Finished, Available, Finished, False)

root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- _birthDate: struct (nullable = true)
 |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |-- address: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- _line: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |   

## 2. Extract FHIR Resources

A FHIR search response is a Bundle; the actual resources are inside `entry[].resource`.


In [10]:
def extract_fhir_resources(raw_df):
    return (
        raw_df
        .select(
            F.explode_outer("entry").alias("entry"),
            "source_file"
        )
        .select(
            F.col("entry.resource").alias("resource"),
            "source_file"
        )
        .filter(F.col("resource").isNotNull())
    )


patient_resources_df = extract_fhir_resources(patient_raw_df)
print(f"Patient resources extracted: {patient_resources_df.count()}")
display(patient_resources_df.limit(5))


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 30, Finished, Available, Finished, False)

Patient resources extracted: 742


SynapseWidget(Synapse.DataFrame, 2851110c-d35d-4f5a-bc60-11ecd802bcae)

## 3. Add Bronze Metadata

Add source path, ingestion date, API request, and extraction/load timestamps for traceability.


In [11]:
def read_raw_metadata():
    return (
        spark.read
        .schema(
            "resource_type string, window_date string, page_number int, "
            "api_url_or_params string, extraction_timestamp string, "
            "data_saved_timestamp string, records_extracted int, "
            "raw_file_path string, status string"
        )
        .json(f"{RAW_METADATA_PATH}/")
    )


def add_bronze_metadata(resources_df, resource_type: str):
    metadata_df = (
        read_raw_metadata()
        .filter(F.lower(F.col("resource_type")) == resource_type)
        .withColumn("extraction_timestamp", F.to_timestamp("extraction_timestamp"))
        .withColumn("data_saved_timestamp", F.to_timestamp("data_saved_timestamp"))
        .alias("m")
    )

    source_df = (
        resources_df
        .withColumn(
            "ingestion_date",
            F.regexp_extract(
                F.col("source_file"),
                r"ingestion_date=([0-9]{4}-[0-9]{2}-[0-9]{2})",
                1
            )
        )
        .withColumn(
            "page_number",
            F.regexp_extract(
                F.col("source_file"),
                r"page_([0-9]+)\.json",
                1
            ).cast("int")
        )
        .withColumn("resource_type", F.lit(resource_type.capitalize()))
        .alias("r")
    )

    return (
        source_df
        .join(
            metadata_df,
            (F.col("r.ingestion_date") == F.col("m.window_date"))
            & (F.col("r.page_number") == F.col("m.page_number")),
            "left"
        )
        .select(
            "r.*",
            F.col("m.api_url_or_params").alias("api_url_or_params"),
            F.col("m.extraction_timestamp").alias("extraction_timestamp"),
            F.col("m.data_saved_timestamp").alias("data_saved_timestamp"),
            F.col("m.records_extracted").alias("raw_page_record_count")
        )
        .withColumn("bronze_loaded_timestamp", F.current_timestamp())
    )


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 31, Finished, Available, Finished, False)

## 4. Write Bronze Delta Tables

Replace only the current ingestion dates so reruns do not append duplicate records.


In [12]:
def write_bronze_table(resource_type: str):
    raw_df = read_raw_resource(resource_type)
    resources_df = extract_fhir_resources(raw_df)
    bronze_df = add_bronze_metadata(resources_df, resource_type)

    table_name = f"bronze_{resource_type}"
    table_path = f"{BRONZE_BASE_PATH}/{table_name}"

    ingestion_dates = [
        row["ingestion_date"]
        for row in bronze_df.select("ingestion_date").distinct().collect()
        if row["ingestion_date"]
    ]

    writer = (
        bronze_df
        .write
        .format("delta")
        .option("overwriteSchema", "true")
    )

    if ingestion_dates:
        predicate = "ingestion_date IN (" + ",".join(
            f"'{date}'" for date in ingestion_dates
        ) + ")"

        (
            writer
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .option("mergeSchema", "true")
            .save(table_path)
        )
    else:
        print(f"No Bronze records found for {resource_type}.")
        return

    print(f"Written Bronze table: {table_name}")
    print(f"Bronze path: {table_path}")
    print(f"Records written: {bronze_df.count()}")


for resource_type in RESOURCE_TYPES:
    print("=" * 70)
    print(f"Processing: {resource_type.upper()}")
    print("=" * 70)
    write_bronze_table(resource_type)


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 32, Finished, Available, Finished, False)

Processing: PATIENT
Written Bronze table: bronze_patient
Bronze path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables/bronze_patient
Records written: 742
Processing: ENCOUNTER
Written Bronze table: bronze_encounter
Bronze path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables/bronze_encounter
Records written: 540
Processing: OBSERVATION
Written Bronze table: bronze_observation
Bronze path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables/bronze_observation
Records written: 97255
Processing: CONDITION
Written Bronze table: bronze_condition
Bronze path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables/bronze_condition
Records written: 15752


## 5. Bronze Validation

Check that each Bronze Delta table contains records.


In [13]:
bronze_validation = []

for resource_type in RESOURCE_TYPES:
    table_name = f"bronze_{resource_type}"
    table_path = f"{BRONZE_BASE_PATH}/{table_name}"
    count = spark.read.format("delta").load(table_path).count()

    bronze_validation.append(
        (table_name, count, "PASS" if count > 0 else "FAIL")
    )

validation_df = spark.createDataFrame(
    bronze_validation,
    ["table_name", "record_count", "status"]
)

display(validation_df)


StatementMeta(, 4e9d5086-6721-497c-bb31-6ff35a4fd612, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a1f6a40-dab5-4e94-b17d-2cad86cc923b)

## Bronze Layer Complete

Expected tables:
- `bronze_patient`
- `bronze_encounter`
- `bronze_observation`
- `bronze_condition`

Next: clean, deduplicate, and apply SCD Type 2 in `03_bronze_to_silver`.
